# Notebook 2 — Preprocessing

**Project:** IntelliSys Ltd. — London Road Collision Severity Prediction  

---

## Objective

Transform the raw London collision dataset (produced in Notebook 1) into a  
clean, model-ready feature matrix. Every preprocessing decision is explicitly  
documented with a business or statistical justification.

**Outputs saved by this notebook:**
- `data/processed/X_train.npy`, `y_train.npy`
- `data/processed/X_val.npy`, `y_val.npy`
- `data/processed/X_test.npy`, `y_test.npy`
- `data/processed/feature_names.txt`
- `outputs/models/scaler.pkl`

---
## Step 1 — Imports and Seeds

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_seeds, outputs_dir
from src.preprocessing import (
    FEATURE_COLUMNS, TARGET_COLUMN, SEVERITY_MAP, CLASS_NAMES,
    select_available_features, remap_target, handle_missing_values,
    encode_time_feature, one_hot_encode,
    split_dataset, scale_features, compute_class_weights
)

np.random.seed(42)
set_seeds(42)

PROCESSED_DIR = project_root / 'data' / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)
print('Setup complete.')

---
## Step 2 — Load the Processed London Dataset

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'london_collisions_2024.csv', low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Confirm target variable is present
assert TARGET_COLUMN in df.columns, f"'{TARGET_COLUMN}' column not found!"
print(f'Target distribution:\n{df[TARGET_COLUMN].value_counts().sort_index()}')

---
## Step 3 — Remap Target Variable to 0-Indexed

PyTorch `CrossEntropyLoss` requires integer class labels in `[0, num_classes-1]`.  
The STATS19 encoding (1=Fatal, 2=Serious, 3=Slight) is remapped as follows:

| STATS19 Code | Meaning | PyTorch Label |
|:---:|:---:|:---:|
| 1 | Fatal | 0 |
| 2 | Serious | 1 |
| 3 | Slight | 2 |

**This mapping must be remembered when interpreting confusion matrices and  
classification reports in Notebooks 3 and 5.**

In [ ]:
df = remap_target(df)
print('Target mapping applied: Fatal=0, Serious=1, Slight=2')
print(df['target'].value_counts().sort_index().rename({0:'Fatal(0)', 1:'Serious(1)', 2:'Slight(2)'}))

---
## Step 4 — Feature Selection

We retain only features available at **prediction time** — i.e. environmental  
and contextual conditions present before the crash outcome is determined.  
Dropped: GPS coordinates, postcodes, administrative reference codes,  
and post-crash outcome fields.

In [ ]:
features = select_available_features(df)
print(f'Feature columns available: {len(features)} / {len(FEATURE_COLUMNS)} requested')
print('\nSelected features:')
for f in features:
    print(f'  {f}')

---
## Step 5 — Encode Time Feature

The raw `time` column contains strings in `HH:MM` format. Rather than one-hot  
encoding 1,440 possible minute values, we extract the hour (0–23).  
This preserves temporal ordering while keeping cardinality manageable.

In [ ]:
df = encode_time_feature(df)

# Update feature list: replace 'time' with 'hour'
features = [f for f in features if f != 'time']
if 'hour' in df.columns:
    features = ['hour'] + features

print(f'Time encoded as hour. Feature count: {len(features)}')

---
## Step 6 — Handle Missing Values

**Decision rules:**
- Columns with **>40% missing**: dropped entirely (documented below).
- **Numeric** columns with moderate missingness: imputed with **median** (robust to outliers).
- **Categorical** columns: imputed with **mode** (most frequent value).

In [ ]:
# Show missing rates for feature columns
missing_summary = pd.DataFrame({
    'feature': features,
    'missing_pct': [df[f].isnull().mean() * 100 for f in features],
    'dtype': [str(df[f].dtype) for f in features]
}).sort_values('missing_pct', ascending=False)

print('Missing value summary for selected features:')
display(missing_summary[missing_summary['missing_pct'] > 0])

In [ ]:
df, features = handle_missing_values(df, features)
print(f'\nFeature count after dropping high-missingness columns: {len(features)}')

---
## Step 7 — One-Hot Encode Categorical Features

Nominal categorical features (road type, casualty type, vehicle type, etc.) are  
one-hot encoded. There is no meaningful ordinal relationship between their values,  
so label encoding would impose a false numeric ordering.

`speed_limit` and `day_of_week` remain numeric (ordinal/quantitative by nature).

The feature count will expand from ~20 columns to ~60–80 binary columns.

In [ ]:
X_df, feature_names = one_hot_encode(df, features)
y = df['target'].values.astype(np.int64)
X = X_df.values.astype(np.float32)

print(f'Feature matrix: {X.shape[0]:,} samples × {X.shape[1]} features')
print(f'(Original {len(features)} → {len(feature_names)} after one-hot encoding)')

---
## Step 8 — Train / Validation / Test Split

**Split: 70% train | 15% validation | 15% test — stratified by class.**

Stratification ensures each split has the same class proportions as the full  
dataset, preventing the fatal class (1%) from being accidentally excluded  
from the test set.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)

print('\nClass distribution across splits:')
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    vals, cnts = np.unique(y_split, return_counts=True)
    pcts = 100 * cnts / len(y_split)
    dist_str = '  '.join([f'{CLASS_NAMES[v]}: {c} ({p:.1f}%)' for v, c, p in zip(vals, cnts, pcts)])
    print(f'  {split_name:5s}  n={len(y_split):6,} | {dist_str}')

---
## Step 9 — Feature Scaling

`StandardScaler` is fitted **on training data only** to prevent data leakage.  
Validation and test sets are transformed using the training statistics.

The fitted scaler is serialised to `outputs/models/scaler.pkl` — it must be  
reloaded and applied identically when the deployed model scores live road data.

In [ ]:
X_train, X_val, X_test, scaler = scale_features(X_train, X_val, X_test)
print('StandardScaler fitted on training data. Val and test sets transformed.')
print(f'Training set — mean: {X_train.mean():.4f}, std: {X_train.std():.4f}')

---
## Step 10 — Class Imbalance: Compute Balanced Class Weights

**Strategy: Weighted CrossEntropyLoss.**

Rather than resampling (SMOTE), we compute balanced class weights and pass  
them to the loss function during training. This approach:
- Is computationally simpler and more reproducible than SMOTE.
- Does not generate synthetic samples that may not reflect real crash dynamics.
- Is mathematically equivalent to upsampling minority classes in expectation.

Higher weight on Fatal (class 0) forces the model to penalise missed fatal  
predictions more heavily — directly addressing the most dangerous error type.

In [ ]:
class_weights = compute_class_weights(y_train)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(CLASS_NAMES, class_weights, color=['#d62728', '#ff7f0e', '#2ca02c'])
ax.set_ylabel('Weight')
ax.set_title('Balanced Class Weights for CrossEntropyLoss', fontweight='bold')
plt.tight_layout()
plt.savefig(project_root / 'outputs' / 'figures' / 'class_weights.png', dpi=150)
plt.show()

for name, w in zip(CLASS_NAMES, class_weights):
    print(f'  {name}: {w:.4f}')

---
## Step 11 — Save Preprocessed Arrays

All arrays are saved to `data/processed/` so downstream notebooks can load  
them without re-running the full preprocessing pipeline.

In [ ]:
np.save(PROCESSED_DIR / 'X_train.npy', X_train)
np.save(PROCESSED_DIR / 'X_val.npy',   X_val)
np.save(PROCESSED_DIR / 'X_test.npy',  X_test)
np.save(PROCESSED_DIR / 'y_train.npy', y_train)
np.save(PROCESSED_DIR / 'y_val.npy',   y_val)
np.save(PROCESSED_DIR / 'y_test.npy',  y_test)
np.save(PROCESSED_DIR / 'class_weights.npy', class_weights)

with open(PROCESSED_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

print('Saved:')
for fname in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'class_weights']:
    path = PROCESSED_DIR / f'{fname}.npy'
    print(f'  {path.name}: {np.load(path).shape}')

print(f'\nFeature names saved to {PROCESSED_DIR / "feature_names.txt"}')
print(f'Scaler saved to outputs/models/scaler.pkl')
print(f'\nInput dimension for MLP: {X_train.shape[1]}')

---
## Preprocessing Summary

| Step | Decision | Justification |
|------|----------|---------------|
| Feature selection | Kept 20+ pre-crash contextual features | Prediction at inference time; no post-crash data |
| Target remapping | 1→0 (Fatal), 2→1 (Serious), 3→2 (Slight) | PyTorch CrossEntropyLoss requires 0-indexed labels |
| Time encoding | HH:MM → hour integer | Preserves temporal order; avoids 1,440-dim one-hot |
| Missing values | Median/mode imputation; drop if >40% missing | Transparent, reproducible; justified per column |
| Categorical encoding | One-hot for nominal features | No false ordinal ordering imposed |
| Class imbalance | Balanced class weights | No synthetic samples; numerically equivalent to oversampling |
| Split | 70/15/15 stratified | Standard academic benchmark; stratification preserves Fatal proportion |
| Scaling | StandardScaler fit on train only | Prevents data leakage; required for MLP convergence |

**Proceed to Notebook 3 — Baseline Model** to establish a Logistic Regression benchmark.